<a href="https://colab.research.google.com/github/misrori/ai/blob/2025/text_to_speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai
!pip install openai pydub


In [ ]:
import openai
import os
from pathlib import Path
from pydub import AudioSegment

# API kulcs beállítása
openai.api_key = "sk-proj-"

def text_to_speech(text, output_file="output.mp3", voice="nova", model="tts-1", speed=1.0):
    """
    Szöveget alakít át beszéddé az OpenAI API segítségével.

    Paraméterek:
        text (str): A hangfájllá alakítandó szöveg
        output_file (str): A kimeneti hangfájl neve/útvonala
        voice (str): A használni kívánt hang (pl. "alloy", "echo", "fable", "onyx", "nova", "shimmer")
        model (str): A használni kívánt modell (pl. "tts-1", "tts-1-hd")
        speed (float): A beszéd sebessége (0.25-től 4.0-ig)
    """
    try:
        response = openai.audio.speech.create(
            model=model,
            voice=voice,
            input=text,
            speed=speed
        )

        # Fájl mentése
        response.stream_to_file(output_file)
        print(f"A hangfájl sikeresen létrehozva: {output_file}")
        return True

    except Exception as e:
        print(f"Hiba történt: {e}")
        return False

def process_long_text(text, final_output="vegso_hangfajl.mp3", temp_dir="temp_hangfajlok", max_chunk_size=4000, **kwargs):
    """
    Hosszú szöveget feldolgoz kisebb darabokban, hangfájlokká alakítja, majd összefűzi őket.

    Paraméterek:
        text (str): A hosszú szöveg
        final_output (str): A végső összefűzött hangfájl neve
        temp_dir (str): Ideiglenes könyvtár a részleteknek
        max_chunk_size (int): Az egyszerre feldolgozható maximális karakterszám
        **kwargs: További paraméterek a text_to_speech funkcióhoz
    """
    # Ideiglenes könyvtár létrehozása, ha nem létezik
    Path(temp_dir).mkdir(parents=True, exist_ok=True)

    # Szöveg felosztása értelmes darabokra mondatok mentén
    # Először töröljük a felesleges szóközöket
    text = text.strip()

    # Szöveg felosztása mondatokra
    sentences = []
    for sentence in text.replace(".", ".@").replace("!", "!@").replace("?", "?@").split("@"):
        if sentence:
            sentences.append(sentence)

    # Mondatok csoportosítása
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        # Ha az aktuális mondat hozzáadása túllépné a megengedett méretet, új chunk kezdése
        if len(current_chunk) + len(sentence) > max_chunk_size:
            chunks.append(current_chunk)
            current_chunk = sentence
        else:
            current_chunk += sentence

    # Utolsó chunk hozzáadása
    if current_chunk:
        chunks.append(current_chunk)

    print(f"A szöveg {len(chunks)} részre lett felosztva feldolgozáshoz.")

    # Chunkok feldolgozása és temp fájlok létrehozása
    temp_files = []
    for i, chunk in enumerate(chunks):
        temp_file = os.path.join(temp_dir, f"temp_{i+1}.mp3")
        success = text_to_speech(chunk, output_file=temp_file, **kwargs)
        if success:
            temp_files.append(temp_file)

    if not temp_files:
        print("Nem sikerült egyetlen hangfájlt sem létrehozni.")
        return

    # Hangfájlok összefűzése
    print("Hangfájlok összefűzése...")
    combined = AudioSegment.empty()
    for temp_file in temp_files:
        audio_segment = AudioSegment.from_mp3(temp_file)
        combined += audio_segment

    # Végső fájl mentése
    combined.export(final_output, format="mp3")
    print(f"Összefűzött hangfájl sikeresen létrehozva: {final_output}")

    # Opcionális: ideiglenes fájlok törlése
    print("Ideiglenes fájlok törlése...")
    for temp_file in temp_files:
        os.remove(temp_file)
    os.rmdir(temp_dir)
    print("Ideiglenes fájlok törölve.")

# Példa használat
if __name__ == "__main__":
    # Egy hosszú példaszöveg
    hosszu_szoveg = """
    Donald Trump amerikai elnök és Vlagyimir Putyin orosz elnök 70 perces telefonbeszélgetése nem hozta meg a várt áttörést az orosz–ukrán konfliktusban. Az Index Frontvonal című műsorában Tarjányi Péter biztonságpolitikai szakértő értékelte a helyzetet, szerinte Oroszország feltételei egyoldalúan előnyösek, és Ukrajna kihagyása a tárgyalásokból súlyos problémákat vet fel.
A beszélgetésben Tarjányi Péter rámutatott, hogy bár az amerikai fél „konstruktívnak” és „pozitívnak” értékelte a megbeszélést, valójában az orosz elnök olyan feltételrendszert vázolt fel, amely szinte kizárólag Moszkva érdekeit szolgálja.

A szakértő szerint különösen aggasztó, hogy miközben a telefonbeszélgetés zajlott, az orosz csapatok tovább folytatták előrenyomulásukat Donyeckben, Zaporizzsja és Luhanszk területen, valamint a kurszki kiszögellésben is. Ez egyértelműen jelzi, hogy Oroszország a tárgyalásokkal párhuzamosan sem lassítja katonai műveleteit – mutatott rá Tarjányi Péter.

Róluk, nélkülük
A Trump és Putyin közötti 70 perces telefonbeszélgetés a hidegháborúból örökölt „forró dróton” zajlott. Az egyeztetésen Putyin határozottan elutasította az általános tűzszünet lehetőségét, ehelyett egy olyan részleges megállapodást javasolt, amely több szempontból is előnyös Oroszország számára.

A tárgyalásokban sem Ukrajna, sem az Európai Unió nem vett részt, ami súlyos aggályokat vet fel a megállapodás végrehajthatóságával kapcsolatban.

Tarjányi Péter kiemelte, hogy a megbeszélés bevezető részében a két elnök kölcsönösen méltatta egymást, mielőtt a konkrét feltételekre tértek volna. A biztonsági szakértő szerint problematikus, hogy míg a tárgyalások folytak, Kijev felett továbbra is szóltak a légvédelmi szirénák, jelezve, hogy a háború érdemi enyhülése egyelőre nem várható.

Tarjányi Péter a műsorban többek között arról is beszélt, hogy

Mit ajánlott valójában Vlagyimir Putyin;
miért problémás az orosz elnök javaslata;
mit jelent a gyakorlatban a tűzszünet;
milyen az Európai Unió helyzete, amely teljesen kimaradt a tárgyalásokból.
Az adásban kitértek a közel-keleti helyzetre is, mivel Izrael újabb jelentős támadást indított a Gázai övezetben, ami több mint 300 halálos áldozattal járt. Ezenkívül érintették Magyarország pozícióját is a nemzetközi színtéren. Elhangzott, hogy az Egyesült Államok nyomást gyakorolt a magyar kormányra, hogy ne vétózza meg az új uniós szankciós csomagot Oroszország ellen. Tarjányi Péter ezt pozitív jelként értékelte, mivel arra utal, hogy az amerikai adminisztráció szakértői szintjén tudatosan kezelik a helyzetet. A szakértő szerint a jelenlegi helyzet alakulása Magyarország külpolitikájára nézve isfontos következményekkel járhat.

    """

    # API kulcs beállítása környezeti változóból (biztonságosabb)
    # os.environ["OPENAI_API_KEY"] = "az_ön_openai_api_kulcsa_ide"

    # Hosszú szöveg feldolgozása és összefűzése
    process_long_text(
        hosszu_szoveg,
        final_output="vegso_hangfajl.mp3",  # Végső összefűzött fájl neve
        temp_dir="temp_hangfajlok",         # Ideiglenes könyvtár
        max_chunk_size=4000,                # Maximum 4000 karakter chunkonként
        voice="nova",                       # Választható: "alloy", "echo", "fable", "onyx", "nova", "shimmer"
        model="tts-1",                      # Választható: "tts-1" vagy "tts-1-hd" (jobb minőség)
        speed=1.0                           # Beszédsebesség (0.25-4.0)
    )

A szöveg 1 részre lett felosztva feldolgozáshoz.


<ipython-input-2-551db4aab076>:29: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_file)


A hangfájl sikeresen létrehozva: temp_hangfajlok/temp_1.mp3
Hangfájlok összefűzése...
Összefűzött hangfájl sikeresen létrehozva: vegso_hangfajl.mp3
Ideiglenes fájlok törlése...
Ideiglenes fájlok törölve.
